# 🧹 Pipeline ETL: De Capa Bronce a Capa Oro
Este cuaderno documenta el paso a paso del tratamiento de datos. El objetivo no es analizar el mercado, sino **limpiar, aplanar y estandarizar** los datos crudos recolectados por los scrapers para que los algoritmos de Machine Learning y el Dashboard de Streamlit puedan consumirlos sin errores.

In [1]:
import sqlite3
import pandas as pd
import ast
import os

# 1. Cargar los datos crudos (Capa Bronce)
# Aquí traemos TODOS los snapshots para no perder el historial de precios
db_cruda = os.path.abspath('inmuebles.db')
conn_cruda = sqlite3.connect(db_cruda)

query_extraccion = """
SELECT
    i.source_name as portal,
    i.external_id,
    s.id as snapshot_id,
    s.precio,
    s.moneda,
    s.ubicacion,
    s.caracteristicas,
    s.raw_extra_data as amenidades,
    s.scraped_at as fecha_registro
FROM inmuebles i
JOIN inmuebles_snapshots s ON i.id = s.inmueble_id
"""

df = pd.read_sql_query(query_extraccion, conn_cruda)
print(f"📦 Total de registros crudos cargados: {len(df)}")
display(df.head(3))

📦 Total de registros crudos cargados: 9824


,portal,external_id,snapshot_id,precio,moneda,ubicacion,caracteristicas,amenidades,fecha_registro
0,mlscaracas,1351436,1,2500.0,USD,{},{},"{""amenidades"": [""Aire acondicionado"", ""Amoblad...",2026-03-24 04:33:40.328993
1,mlscaracas,1351543,2,1800.0,USD,{},{},"{""amenidades"": [""Aire acondicionado"", ""Amoblad...",2026-03-24 04:33:40.330379
2,mlscaracas,1578391,3,3500.0,USD,{},{},"{""amenidades"": [""Aire acondicionado"", ""Alarma""...",2026-03-24 04:33:40.331755


## Paso 1: Aplanamiento de JSONs (Flattening)
Las columnas `ubicacion` y `caracteristicas` contienen diccionarios en formato texto. Debemos expandirlos en columnas nativas tabulares.

In [2]:
# Función segura de parseo
def parse_json_column(texto):
    if pd.isna(texto) or texto == "": return {}
    try: return ast.literal_eval(str(texto))
    except: return {}

# Extraemos ubicación
df['ubicacion_dict'] = df['ubicacion'].apply(parse_json_column)
df_ubi = df['ubicacion_dict'].apply(pd.Series)

# Extraemos características
df['carac_dict'] = df['caracteristicas'].apply(parse_json_column)
df_carac = df['carac_dict'].apply(pd.Series)

# Unimos todo y eliminamos las columnas basura
df_plano = pd.concat([df, df_ubi, df_carac], axis=1)
df_plano = df_plano.drop(columns=['ubicacion', 'caracteristicas', 'ubicacion_dict', 'carac_dict'])

print("✅ Columnas expandidas exitosamente.")
display(df_plano.head(3))

✅ Columnas expandidas exitosamente.


,portal,external_id,snapshot_id,precio,moneda,amenidades,fecha_registro,municipio,urbanismo,m2_totales,habitaciones,banos
0,mlscaracas,1351436,1,2500.0,USD,"{""amenidades"": [""Aire acondicionado"", ""Amoblad...",2026-03-24 04:33:40.328993,NaN,NaN,NaN,NaN,NaN
1,mlscaracas,1351543,2,1800.0,USD,"{""amenidades"": [""Aire acondicionado"", ""Amoblad...",2026-03-24 04:33:40.330379,NaN,NaN,NaN,NaN,NaN
2,mlscaracas,1578391,3,3500.0,USD,"{""amenidades"": [""Aire acondicionado"", ""Alarma""...",2026-03-24 04:33:40.331755,NaN,NaN,NaN,NaN,NaN


In [3]:
# Función segura de parseo
def parse_json_column(texto):
    if pd.isna(texto) or texto == "": return {}
    try: return ast.literal_eval(str(texto))
    except: return {}

# Extraemos ubicación
df['ubicacion_dict'] = df['ubicacion'].apply(parse_json_column)
df_ubi = df['ubicacion_dict'].apply(pd.Series)

# Extraemos características
df['carac_dict'] = df['caracteristicas'].apply(parse_json_column)
df_carac = df['carac_dict'].apply(pd.Series)

# Unimos todo y eliminamos las columnas basura
df_plano = pd.concat([df, df_ubi, df_carac], axis=1)
df_plano = df_plano.drop(columns=['ubicacion', 'caracteristicas', 'ubicacion_dict', 'carac_dict'])

print("✅ Columnas expandidas exitosamente.")
display(df_plano.head(3))

✅ Columnas expandidas exitosamente.


,portal,external_id,snapshot_id,precio,moneda,amenidades,fecha_registro,municipio,urbanismo,m2_totales,habitaciones,banos
0,mlscaracas,1351436,1,2500.0,USD,"{""amenidades"": [""Aire acondicionado"", ""Amoblad...",2026-03-24 04:33:40.328993,NaN,NaN,NaN,NaN,NaN
1,mlscaracas,1351543,2,1800.0,USD,"{""amenidades"": [""Aire acondicionado"", ""Amoblad...",2026-03-24 04:33:40.330379,NaN,NaN,NaN,NaN,NaN
2,mlscaracas,1578391,3,3500.0,USD,"{""amenidades"": [""Aire acondicionado"", ""Alarma""...",2026-03-24 04:33:40.331755,NaN,NaN,NaN,NaN,NaN


## Paso 2: Feature Engineering (Ingeniería de Características)
La columna `amenidades` contiene listas de texto libre. Vamos a crear variables binarias (1 o 0) para las amenidades más críticas que definen el precio en Caracas: **Pozo de agua**, **Planta eléctrica** y **Piscina**.

In [4]:
# Convertimos todo a minúsculas para facilitar la búsqueda
df_plano['amenidades_str'] = df_plano['amenidades'].astype(str).str.lower()

# Creamos variables binarias (Dummy variables)
df_plano['tiene_pozo'] = df_plano['amenidades_str'].apply(lambda x: 1 if 'pozo' in x else 0)
df_plano['tiene_planta'] = df_plano['amenidades_str'].apply(lambda x: 1 if 'planta' in x else 0)
df_plano['tiene_piscina'] = df_plano['amenidades_str'].apply(lambda x: 1 if 'piscina' in x else 0)

# Borramos la columna de texto original
df_plano = df_plano.drop(columns=['amenidades', 'amenidades_str'])

print(f"💧 Inmuebles con Pozo detectados: {df_plano['tiene_pozo'].sum()}")
print(f"⚡ Inmuebles con Planta detectados: {df_plano['tiene_planta'].sum()}")

💧 Inmuebles con Pozo detectados: 955
⚡ Inmuebles con Planta detectados: 996


## Paso 3: Limpieza de Tipos de Datos y Valores Nulos
Los algoritmos de Machine Learning odian los valores de texto en columnas numéricas y los nulos (`NaN`). Vamos a forzar los tipos numéricos correctos.

In [5]:
# Nos aseguramos de que existan las columnas (por si un scraper falló)
cols_esperadas = ['m2_totales', 'habitaciones', 'banos']
for col in cols_esperadas:
    if col not in df_plano.columns:
        df_plano[col] = None

# Forzamos conversión numérica (lo que sea error, se vuelve NaN temporalmente)
df_plano['m2_totales'] = pd.to_numeric(df_plano['m2_totales'], errors='coerce')
df_plano['habitaciones'] = pd.to_numeric(df_plano['habitaciones'], errors='coerce')
df_plano['banos'] = pd.to_numeric(df_plano['banos'], errors='coerce')

# Opcional: Rellenar nulos de características con 0 (o dejarlos como NaN para imputarlos después en ML)
# Para la base de datos Oro, es mejor mantenerlos nulos (NULL en SQL) si realmente no existen
# para no sesgar promedios, pero los aseguramos como tipo float.

# Limpiar texto del municipio (Mayúscula inicial, quitar espacios extra)
if 'municipio' in df_plano.columns:
    df_plano['municipio'] = df_plano['municipio'].astype(str).str.strip().str.title()
    # Reemplazar 'Nan' o 'None' por nulo real
    df_plano.loc[df_plano['municipio'].isin(['Nan', 'None']), 'municipio'] = None

print("✅ Tipos de datos estandarizados.")
df_plano.info()

✅ Tipos de datos estandarizados.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9824 entries, 0 to 9823
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   portal          9824 non-null   object 
 1   external_id     9824 non-null   object 
 2   snapshot_id     9824 non-null   int64  
 3   precio          9594 non-null   float64
 4   moneda          9824 non-null   object 
 5   fecha_registro  9824 non-null   object 
 6   municipio       9791 non-null   object 
 7   urbanismo       9771 non-null   object 
 8   m2_totales      2149 non-null   float64
 9   habitaciones    1829 non-null   float64
 10  banos           527 non-null    float64
 11  tiene_pozo      9824 non-null   int64  
 12  tiene_planta    9824 non-null   int64  
 13  tiene_piscina   9824 non-null   int64  
dtypes: float64(4), int64(4), object(6)
memory usage: 1.0+ MB


## Paso 4: Exportación a Capa Oro (`inmuebles_clean.db`)
Finalmente, guardamos este DataFrame perfecto y estructurado en una nueva base de datos que será la ÚNICA fuente de verdad para el Dashboard y los Modelos de ML.

In [6]:
# Nos conectamos a la nueva base de datos (se creará automáticamente)
db_limpia = os.path.abspath('inmuebles_clean.db')
conn_limpia = sqlite3.connect(db_limpia)

# Guardamos el dataframe como una tabla SQL
# 'replace' asegura que si volvemos a correr el script, se sobrescribe con la versión más fresca
df_plano.to_sql('inmuebles_limpios', conn_limpia, if_exists='replace', index=False)

conn_cruda.close()
conn_limpia.close()

print(f"🚀 ¡Datos limpios guardados exitosamente en {db_limpia}!")

🚀 ¡Datos limpios guardados exitosamente en /Users/chesterdarial/PycharmProjects/AlquilerCaracas/inmuebles_clean.db!
